## 1. 今日の量子コンピュータの問題

- Noisy Intermediate-Scale Quantum (NISQ) デバイス
    - 量子回路が深くなる（ゲート数が多くなる）ほど、誤差が大きくなる
    - 十分な量子ビット数ではない
- 量子デバイスは特別な量子ゲート演算しか用意されていない  
  Quantinuum H-series: , IBM Quantum:
- 超電導型量子デバイスでは特定の２量子ビット間の量子ビット演算しか用意されていない
- それぞれの量子デバイスに対して、量子ソフトウェアツールキットが用意されてる


### 1-1. TKETとは
- Quantum Software Development Kit
- TKETに実装されている回路最適化はC++で実装
- pythonモジュール　`pytket`で利用可能
- 最適化コンパイラ：　ユーザーフレンドリーな回路→量子デバイスで実行可能な回路に変換可能
    - Language-agnostic (多くの量子プログラミングフレームワーク(qiskit, Cirq, etc)をサポート)
    - Retagetable (多くの量子デバイス(IBM, Quantinuum, Amazon Braket(IonQ, Rigetti, IQM) etc)をサポート)
    - Circuit Optimisation (量子計算時に生じるデバイスエラーの影響を最小化。デバイス依存＆デバイス非依存のものが実装)
    
<img src="./fig/tket1.png" width="750">



#### 参照
- [pytket ドキュメント](https://docs.quantinuum.com/tket/api-docs/)
- [pytket ユーザーガイド](https://docs.quantinuum.com/tket/user-guide/)
- [t|ket⟩ : A Retargetable Compiler for NISQ Devices](https://arxiv.org/abs/2003.10611)
- [TKET slack channel](https://join.slack.com/t/tketusers/shared_invite/zt-2aoan2s87-WDdZQeY2dbJQgAQE6O~3qg)

<img src="./fig/slack-qr.png" width="250">


### 1-2. pytketと拡張 pytket (python パッケージ)
Python 3.11,12で動作確認をしています。

|  パッケージ |  概要  |
| :---- | :---- |
|  pytket  |  TKETを利用するためのpython モジュール  ( available for python3.10 or higher )|
|  pytket-quantinuum  |  Quantinuumデバイス、エミュレータを利用するためのpytket-extension  |
|  pytket-qiskit  |  qiskit、IMBQデバイスを利用するためのpytket-extension  |
|  pytket-braket  |  Amazon Braketを利用するためのpytket-extension  |
|  pytket-circ    |  Google circを利用するためのpytket-extension  |
|  pytket-qulacs  |  Qulacsシミュレータを利用するためのpytket-extension  |

<img src="./fig/tket2.png" width="850">

In [ ]:
#!pip install pytket-qiskit
#!pip install -U pytket-quantinuum
#!pip install -U pytket-quantinuum[pecos]
!pip install -U pytket-qulacs

In [ ]:
!pip freeze | grep pytket

## 2. 量子回路を作成する　（より詳しい内容は２日目に行います）
ここでは IBMの量子デバイスやシュミレーションを利用できる`qiskit`と`TKET`でBell状態を作成する。

### 2-1. `qiskit`でBell状態を作成
$$ |\Psi\rangle = \frac{1}{\sqrt{2}}(|00\rangle+|11\rangle)$$

In [ ]:
# qiskitを使って量子回路を作成するために必要なモジュールをインポート
from qiskit import QuantumCircuit

# 2 qubit から成る空の量子回路を作成
qs_bell = QuantumCircuit(2)

#ユニタリゲートを回路に追加。qubitのindexは0,1,...
qs_bell.h(0) #q_0に作用するHadamard gateを追加
qs_bell.cx(0,1) #q_0を制御ビット、q_1を標的ビットとするCNOT gateを回路に追加

#(パウリZの基底での)測定
qs_bell.measure_all()

#回路の描画
qs_bell.draw(output='mpl')

In [ ]:
#from qiskit.tools.visualization import circuit_drawer
#circuit_drawer(qs_bell, output='mpl')

### 2-2. IBMが提供しているローカルシミュレータで計算

In [ ]:
from qiskit_aer import Aer #量子回路を実行するシミュレータを利用するために必要
from qiskit.visualization import plot_histogram #量子計算では最後に測定を行うが、測定結果を視覚化するために必要

In [ ]:
#Aer.backends()

In [ ]:
#シミュレータの指定
ibm_sim = Aer.get_backend('aer_simulator')

In [ ]:
#回路の実行
handle = ibm_sim.run(qs_bell, shots=1000)

#実行した結果それぞれの測定結果が何回ずつ得られたか
counts = handle.result().get_counts()
plot_histogram(counts)

### 2-3. `TKET`でBell状態を作成

In [ ]:
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter

# 2 qubit から成る空の量子回路を作成
bell = Circuit(2)

#ユニタリゲートを回路に追加
bell.H(0).CX(0,1)

#(パウリZの基底での)測定
bell.measure_all()

#回路の描画
render_circuit_jupyter(bell)

Note：TKETの可視化では、可視化した量子回路の画像ファイルを出力できる。

### 2-4. `pytket-qiskit`でTKET 量子回路をIBMのローカルシミュレータで計算

In [ ]:
from pytket.extensions.qiskit import AerBackend

#シミュレータの指定
backend = AerBackend()

#回路の実行
handle = backend.process_circuit(bell, n_shots =1000)

#実行した結果、それぞれの測定結果が何回ずつ得られたかの取得
result_counts = backend.get_result(handle).get_counts()

#これをヒストグラム化
plot_histogram(result_counts)

### 2-5. `pytket-qiskit`でTKET 量子回路をIBMの量子デバイスで計算

IBM tokenの設定

In [ ]:
# Replace the placeholders with your actual values
ibm_token = '<your_ibm_token_here>'
inst = '<your_instance_CRN_here>'

In [ ]:
path = 'ibm-token'
f = open(path)
ibm_token = f.read()
f.close()
#from pytket.extensions.qiskit.backends.config import set_ibmq_config
#set_ibmq_config(ibmq_api_token=ibm_token, instance=f"ibm-q/open/main")

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=ibm_token, overwrite=True)

In [ ]:
from pytket.extensions.qiskit import IBMQBackend, IBMQEmulatorBackend
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter

In [ ]:
#利用可能なデバイスの確認
device = IBMQBackend.available_devices()
[dev.device_name for dev in device]

In [ ]:
#シミュレータを指定したのと全く同じように量子コンピュータを指定
# ibm_backend = AerBackend()
#ibm_backend = IBMQBackend("ibm_pittsburgh")
ibm_backend = IBMQEmulatorBackend("ibm_pittsburgh")

### IBMの量子デバイス（ibm_pittsburgh）にジョブを実行

IBM Quantum device の情報  
https://quantum.ibm.com/services/resources

In [ ]:
#上で指定したデバイスで回路を実行するために回路をコンパイルする。コンパイルされた回路は指定したバックエンドごとに変わります。
#(コンパイル前後の回路を図示して比較してみましょう！)
ibm_bell = ibm_backend.get_compiled_circuit(bell)
render_circuit_jupyter(ibm_bell)
render_circuit_jupyter(bell)

In [ ]:
#指定したデバイス上で回路を実行
handle = ibm_backend.process_circuit(ibm_bell, n_shots =1000)

In [ ]:
result = ibm_backend.get_result(handle)
counts = result.get_counts()
plot_histogram(counts)

### 2-5. `pytket-quantinuum`でTKET 量子回路をQuantinuum エミュレータで計算

In [ ]:
from pytket.extensions.quantinuum import QuantinuumAPIOffline
api = QuantinuumAPIOffline()

In [ ]:
from pytket.extensions.quantinuum import QuantinuumBackend

#利用可能なバックエンドの確認
QuantinuumBackend.available_devices(api_handler = api)

In [ ]:
# 回路を実行するエミュレータの指定。ここではノイズのないローカルエミュレータを指定。
quantinuum_backend = QuantinuumBackend(device_name ='H1-1LE',api_handler = api)

# 回路を実行するためのコンパイル
quantinuum_bell = quantinuum_backend.get_compiled_circuit(bell)

#コンパイル前後での回路の違いを描画して確認
render_circuit_jupyter(quantinuum_bell)
render_circuit_jupyter(bell)

In [ ]:
# 回路の実行 (左辺の変数に実行結果が格納される)
handle = quantinuum_backend.process_circuit(quantinuum_bell, n_shots=1000)
# 結果の取得
result = quantinuum_backend.get_result(handle)

In [ ]:
# それぞれの測定結果が得られた回数を取得
counts = result.get_counts()
plot_histogram(counts)

### 2-6. `pytket-qulacs`でTKET 量子回路をQulacsシミュレータで計算

In [ ]:
from pytket.extensions.qulacs import QulacsBackend

#回路を実行するシミュレータとしてQslacsシミュレータを指定する
qulacs_backend = QulacsBackend()

In [ ]:
qulacs_backend.backend_info

In [ ]:
#回路の実行
handle = qulacs_backend.process_circuit(bell, n_shots =1000)
#測定結果の取得
result_counts = qulacs_backend.get_result(handle).get_counts()
plot_histogram(result_counts)

### GPU上でQulacsを利用している場合にも対応している


In [ ]:
#from pytket.extensions.qulacs import QulacsGPUBackend
#qualcs_backend = QulacsGPUBackend()

#handle = qulacs_backend.process_circuit(bell, n_shots =1000)
#result_counts = qulacs_backend.get_result(handle).get_counts()
#plot_histogram(result_counts)

詳しくは
https://tket.quantinuum.com/extensions/pytket-qulacs/
を参照ください

## 3. 量子回路の最適化
量子計算時に生じるデバイスエラーの影響を最小化。  
デバイス非依存の最適化とデバイス依存の最適化（実はすでに上記で利用）がある。  
詳２くは三日目にご紹介します。

### 3-1. `PauliSquash` 関数を利用した、量子回路の最適化
TKETには量子回路を最適化する様々な機能が用意されている。
ここで`PauliSquash` 関数を利用した回路の最適化（デバイス非依存）を行う。
`PauliSquash` 関数：Pauli ゲートとCliffordゲートで表現された量子回路を出力）

ランダムな量子回路を作成し、回路の深さとCXの深さを数える。

In [ ]:
from pytket.pauli import Pauli
from pytket.circuit import PauliExpBox, fresh_symbol, OpType
from pytket.passes import DecomposeBoxes
box = PauliExpBox([Pauli.I, Pauli.Z, Pauli.X, Pauli.Y], fresh_symbol('tm'))
from pytket.utils import Graph
import numpy as np

def get_random_pauli_gadgets(n_qubits, n_pauli_gadgets, max_entangle):
    """ランダムにユニタリゲートを追加したパラメタ付き量子回路を準備する関数"""
    """ n_qubits個のqubitから成る回路に最大n_pauli_gadgets個のPauliExpBox(expの型に、ここでは長さmax_entangleのPauli stringが乗ったもの)を追加したものを準備し、そのPauliExpBoxを分解した回路を返している"""
    paulis = [Pauli.I, Pauli.X, Pauli.Y, Pauli.Z]
    circ = Circuit(n_qubits)
    for i in range(n_pauli_gadgets):
        ls_paulis = [np.random.choice(paulis) for k in range(max_entangle)]
        if ls_paulis.count(Pauli.Y) % 2 == 0:
            continue
        if len(ls_paulis) - ls_paulis.count(Pauli.I) <= 1:
            continue
        qubits = np.random.choice(
            [i for i in range(n_qubits)], size=max_entangle, replace=False
        )
        box = PauliExpBox(ls_paulis, fresh_symbol('a'))
        circ.add_pauliexpbox(box, sorted(qubits))
    DecomposeBoxes().apply(circ)
    return circ

ランダムな量子ゲート（Pauliガジェット）を含んだ量子回路を作成

In [ ]:
circ = get_random_pauli_gadgets(
    n_qubits=8, n_pauli_gadgets=300, max_entangle=5
)
print('Circuit depth: ', circ.depth())
print('CX depth: ', circ.depth_by_type(OpType.CX))
render_circuit_jupyter(circ)

`PauliSquash` 関数を使って、量子回路の最適化

In [ ]:
# Circuit optimization by using compiler passes.
from pytket.passes import PauliSquash
circx = circ.copy()
PauliSquash().apply(circx)
#FullPeepholeOptimise().apply(circx)
print('Circuit depth: ', circx.depth())
print('CX depth: ', circx.depth_by_type(OpType.CX))
render_circuit_jupyter(circx)

## 4. 量子回路の変換
pytketでは
- qiskitで記述した量子回路(`qiskit.QuantumCircuit`)からTKETの量子回路のクラスに変換が可能
- TKETで記述した量子回路からqiskitの量子回路(`qiskit.QuantumCircuit`)のクラスに変換が可能
- TKETで記述した量子回路と他の量子プログラミング言語(一部)での量子回路の交換が可能

参照：[pytket-extensions](https://tket.quantinuum.com/api-docs/extensions.html) 

### 4-1. `qiskit`の量子回路から`TKET`の量子回路に変換

In [ ]:
from pytket.extensions.qiskit import qiskit_to_tk

In [ ]:
bell_2 = qiskit_to_tk(qs_bell)
bell_2

In [ ]:
render_circuit_jupyter(bell_2)

### 4-2. `TKET`の量子回路から`qiskit`の量子回路に変換

In [ ]:
from pytket.extensions.qiskit import tk_to_qiskit

In [ ]:
qs_bell_2 = tk_to_qiskit(bell)
qs_bell_2

In [ ]:
qs_bell_2.draw()

弊社Quantinuumのご紹介
- Quantinuum ウェブサイト（ 英語 ）： https://www.quantinuum.com/
- Quantinuum K.K. ウェブサイト（ 日本語 ）： https://quantinuum.co.jp/
- ニュース（ 日本語 ）： https://quantinuum.co.jp/news/  
- X（ 日本語 ）： https://x.com/quantinuum_jp?lang=en
- 採用情報（ 英語 ）：https://www.quantinuum.com/careers
- TKET slack channel：[TKET slack channel](https://join.slack.com/t/tketusers/shared_invite/zt-2aoan2s87-WDdZQeY2dbJQgAQE6O~3qg)

<img src="./fig/slack-qr.png" width="250">
